# 06 — Agent Tools: the conversational Analyst

Cognition ships a Strands-powered `Analyst` class with 9 tools that give an LLM structured access to the cassette store. The agent translates natural-language questions into DSL calls, cites the exact sources returned, and refuses to fill in from training-world knowledge when the corpus can't answer.

| Tool | Purpose |
|---|---|
| `set_schema` | Activate an ontology (dict-first, no temp files) |
| `ingest` | Delta ingest + auto-run extraction diagnostics |
| `reingest` | Re-extract all known docs under the active schema |
| `extraction_report` | Coverage + dead anchors + overfit objects |
| `ask` | Single-claim verdict with cited sources |
| `connect` | Multi-hop connectivity — SUPPORTS / REFUTES / NEI |
| `any_of` | One tree walk, N targets, ranked verdicts |
| `record_finding` | Cross-session memory (persists under `<root>/findings/`) |
| `list_findings` | Read prior investigations — the store remembers |

The system prompt enforces three non-negotiables:

- Never output a verdict without citing `ask()` sources.
- When θ > 0.7, explicitly say the corpus doesn't answer the question. Do NOT fill in from training knowledge.
- Check `list_findings()` first; run `extraction_report()` before answering on a new store.

## Setup: a small cassette store

Build the store first so the agent has something to read. The `Analyst` takes the store as its sole required constructor argument — everything else (schema, documents, findings) flows through the agent's tools.

In [ ]:
import json, tempfile, os
from cognition.cassette import InfonStore

SCHEMA = {
    "toyota":    {"type": "actor",    "tokens": ["toyota"]},
    "honda":     {"type": "actor",    "tokens": ["honda"]},
    "tesla":     {"type": "actor",    "tokens": ["tesla"]},
    "panasonic": {"type": "actor",    "tokens": ["panasonic"]},
    "catl":      {"type": "actor",    "tokens": ["catl"]},
    "bmw":       {"type": "actor",    "tokens": ["bmw"]},
    "ford":      {"type": "actor",    "tokens": ["ford"]},
    "invest":    {"type": "relation", "tokens": ["invest", "invested", "investment"]},
    "partner":   {"type": "relation", "tokens": ["partner", "partners", "partnered"]},
    "supply":    {"type": "relation", "tokens": ["supply", "supplies", "supplier"]},
    "acquire":   {"type": "relation", "tokens": ["acquire", "acquired"]},
    "batteries":   {"type": "feature", "tokens": ["battery", "batteries"]},
    "solid_state": {"type": "feature", "tokens": ["solid-state", "solid state"]},
}

tmpdir = tempfile.mkdtemp(prefix="cognition_06_")
schema_path = os.path.join(tmpdir, "schema.json")
with open(schema_path, "w") as f:
    json.dump(SCHEMA, f)

DOCS = [
    {"id": "d01", "timestamp": "2026-01-05",
     "text": "Toyota invested in solid-state technology."},
    {"id": "d02", "timestamp": "2026-01-20",
     "text": "Toyota partnered with Panasonic."},
    {"id": "d03", "timestamp": "2026-02-01",
     "text": "Panasonic supplies CATL."},
    {"id": "d04", "timestamp": "2026-02-15",
     "text": "Honda partnered with CATL."},
    {"id": "d05", "timestamp": "2026-03-01",
     "text": "Tesla invested in batteries."},
    {"id": "d06", "timestamp": "2026-03-15",
     "text": "Ford partnered with BMW."},
]

store = InfonStore(os.path.join(tmpdir, "store"), schema_path=schema_path)
r = store.ingest(DOCS)
print(f"ingested {len(r['ingested'])} docs, {r['n_infons']} infons")
print(r["report"].summary())

## Create an Analyst

The `Analyst` wraps a Strands `Agent` around the store's tools. Pin a Bedrock model so results are reproducible — Strands' default model ID can rotate as Bedrock's model catalog updates.

`stream=False` silences the token-by-token streaming handler so `chat()` returns the full response as a single string, suitable for scripted use. Leave the default on for live REPL sessions.

```python
from cognition.cassette import Analyst
from strands.models.bedrock import BedrockModel

model = BedrockModel(
    model_id="us.anthropic.claude-sonnet-4-5-20250929-v1:0",
    region_name="us-west-2",
)
a = Analyst(store, model=model, stream=False)
```

The rest of this notebook shows what the agent does with each tool — first directly (no LLM), then via a live Bedrock call. If you don't have Bedrock access, stop after the direct-tool section — you've already seen everything that costs money.

## Tool 1 — `extraction_report`: coverage first, queries second

When the agent picks up a store it's never seen, the first thing it does is check coverage. A corpus where half the docs extracted nothing is a signal to edit the schema before running queries.

The tool returns the same `ExtractionReport` the store's `ingest()` attaches — four failure-mode categories, each with concrete examples.

In [ ]:
report = store.extraction_report()
print(report.summary())

## Tool 2 — `ask`: single-claim verdict with cited sources

The agent translates an English question like *"Does Toyota invest in solid-state?"* into a triple (`subject=toyota, predicate=invest, object=solid_state`) and calls `store.ask()`. Below we call the store directly so you see the shape of the data the agent reasons over.

In [ ]:
from cognition.cassette import Query

v = store.ask(Query().where(subject="toyota", predicate="invest",
                             object="solid_state"))
print(f"verdict:  {v.label}")
print(f"S={v.mass.supports:.2f}  R={v.mass.refutes:.2f}  \u03b8={v.mass.theta:.2f}")
print(f"sources ({len(v.sources)}):")
for s in v.sources[:3]:
    print(f"  \u2022 {s.sentence}   (conf={s.confidence:.2f})")

## Tool 3 — `connect`: multi-hop chains

*"Is Toyota connected to CATL?"* — agent picks `connect()` instead of `ask()` because there's no direct infon with both endpoints. MCTS follows connective edges (partner / supply / acquire) up to the `max_hops` cap.

In [ ]:
CONNECTIVE = {"partner", "supply", "acquire", "license", "invest"}
v = store.connect("toyota", "catl", connective_predicates=CONNECTIVE)
print(f"toyota \u2192 catl: {v.label}  S={v.mass.supports:.2f}  \u03b8={v.mass.theta:.2f}")
for s in v.sources:
    print(f"  {s.subject} \u2192 {s.predicate} \u2192 {s.object}")

## Tool 4 — `any_of`: one tree walk, many targets

*"Which of Toyota's peers are connected to CATL?"* — rather than N separate `connect()` calls, `any_of` does one MCTS walk and assigns each target its own verdict. Cheaper and more consistent.

In [ ]:
vs = store.any_of("toyota", {"catl", "panasonic", "honda", "bmw", "tesla"},
                  connective_predicates=CONNECTIVE)
for t, v in sorted(vs.items(), key=lambda kv: -kv[1].mass.supports):
    print(f"  {t:<10} {v.label:<18}  S={v.mass.supports:.2f}  "
          f"\u03b8={v.mass.theta:.2f}")

## Tools 5 & 6 — `record_finding` / `list_findings`: cross-session memory

At the end of a meaningful investigation, the agent calls `record_finding()` to write a synthesized note to `<root>/findings/<id>.json`. The next session that opens the same store reads these with `list_findings()` and references them instead of re-deriving.

This is the piece that makes the store feel stateful across sessions — the *raw facts* are in cassettes; the *learned conclusions* are in findings.

In [ ]:
f = store.record_finding(
    title="Toyota supply chain reaches CATL via Panasonic",
    body=("Toyota \u2192 partner \u2192 Panasonic \u2192 supply \u2192 CATL. "
           "Two-hop chain, both hops at conf \u2265 0.85. No retractions in "
           "the corpus as of snapshot 2026-03-15."),
    tags=["supply_chain", "toyota", "catl"],
    cites=[{"triple": "toyota/partner/panasonic", "doc_id": "d02"},
           {"triple": "panasonic/supply/catl",    "doc_id": "d03"}],
)
print(f"recorded: {f.id}")

# A later session opens the store and reads the finding back.
for f in store.findings(limit=5):
    print(f"  \u2022 {f.title}   (tags: {f.tags})")

## Bring it all together with a live Analyst

The cells below require `strands-agents`, `boto3`, and Bedrock access in your AWS account. Skip to the summary if you don't have one of these — you've already seen every call the agent will make; Strands just routes them via a model.

Four natural-language questions, four different tool choices. The agent:

1. Calls `list_findings()` first (per system prompt) to see past conclusions.
2. Calls `extraction_report()` to check the corpus is healthy.
3. Routes *"Does Toyota invest in solid-state?"* to `ask`.
4. Routes *"Is Toyota connected to CATL?"* to `connect`.
5. Routes *"Which of these companies is OpenAI linked to?"* to `any_of`.
6. Refuses to answer questions the corpus can't support (θ > 0.7 → honest NEI).

In [ ]:
# Uncomment to run with a live Bedrock Agent.
#
# from cognition.cassette import Analyst
# from strands.models.bedrock import BedrockModel
#
# model = BedrockModel(
#     model_id="us.anthropic.claude-sonnet-4-5-20250929-v1:0",
#     region_name="us-west-2",
# )
# a = Analyst(store, model=model, stream=False)
#
# print(a("Does Toyota invest in solid-state battery technology?"))
# print(a("Is Toyota connected to CATL through any supply chain?"))
# print(a("Which of {catl, panasonic, honda, bmw, tesla} is Toyota linked to?"))
# print(a("Does Tesla have an acquisition agreement with CATL?"))  # θ \u2192 1

print("Skipping live Bedrock call. Uncomment the block above when you have\n"
       "AWS credentials + Bedrock model access configured.")

## Summary

| What you saw | Where it lives |
|---|---|
| 9 tools backing the conversational layer | `cognition/src/cognition/cassette/analyst.py` |
| System prompt enforcing source citation + honest NEI | same file, `SYSTEM_PROMPT` constant |
| Coverage diagnostic surfaced before queries | `extraction_report()` from `ingest()` |
| Single-claim, multi-hop, one-of-many as three distinct tool shapes | `ask` / `connect` / `any_of` |
| Cross-session memory via `findings/<id>.json` | `store.record_finding` + `store.findings` |

**Next:**
- **[07 — Cloud](07_cloud.ipynb)**: cassettes on S3, ingest fan-out, Lambda container deployment.
- **[08 — Category Theory](08_category_theory.ipynb)**: Kan-based schema migration with the functor dict.
- **12 — Sheaf GNN** (new): the trained prior that lives under `<root>/_model/gnn.pt`.

In [ ]:
import shutil
shutil.rmtree(tmpdir)
print("Done.")